# B. 공식 validation 기준선·평가

C1/C2와 같은 공식 train/validation 분할과 26차원 전처리를 사용한다. 기본 실행에서는 `test_final.csv`를 읽지 않는다.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'data' / 'final' / 'train_clean.csv').exists():
            return candidate
    raise FileNotFoundError('data/final/train_clean.csv이 있는 프로젝트 루트를 찾을 수 없습니다.')

ROOT = find_project_root()
FINAL_DIR = ROOT / 'data' / 'final'
RESULT_ROOT = ROOT / 'outputs' if (ROOT / 'outputs').exists() else ROOT / 'results' / 'B'
SCRIPT_DIR = ROOT / 'scripts' if (ROOT / 'scripts').exists() else ROOT / 'members' / 'B'
TRAIN_PATH = FINAL_DIR / 'train_clean.csv'
VALIDATION_PATH = FINAL_DIR / 'validation_final.csv'
print('Project root:', ROOT)

## 1. 공식 개발 데이터 확인

In [ ]:
train = pd.read_csv(TRAIN_PATH)
validation = pd.read_csv(VALIDATION_PATH)
summary = pd.DataFrame({
    'rows': [len(train), len(validation)],
    'features': [train.shape[1] - 1, validation.shape[1] - 1],
    'duplicates': [train.duplicated().sum(), validation.duplicated().sum()],
    'missing': [train.isna().sum().sum(), validation.isna().sum().sum()],
    'purchase_count': [train['Revenue'].astype(int).sum(), validation['Revenue'].astype(int).sum()],
    'purchase_rate': [train['Revenue'].astype(int).mean(), validation['Revenue'].astype(int).mean()],
}, index=['train', 'validation'])
display(summary.style.format({'purchase_rate': '{:.2%}'}))

## 2. 공식 validation 기준선 실행

로지스틱 회귀와 랜덤포레스트를 공식 train에서 학습하고 공식 validation에서 평가한다.

In [ ]:
import sys
scripts_directory = str(SCRIPT_DIR)
if scripts_directory not in sys.path:
    sys.path.insert(0, scripts_directory)
from run_baseline import main
main(evaluate_test=False)

## 3. 기준 성능과 임계값 후보

In [ ]:
metrics = pd.read_csv(RESULT_ROOT / 'metrics' / 'validation_baseline_metrics.csv')
threshold_candidates = pd.read_csv(RESULT_ROOT / 'metrics' / 'validation_threshold_candidates.csv')
formatters = {
    'threshold': '{:.2f}', 'accuracy': '{:.3f}', 'precision': '{:.3f}',
    'recall': '{:.3f}', 'f1': '{:.3f}', 'roc_auc': '{:.3f}',
    'average_precision': '{:.3f}'
}
display(metrics.style.format(formatters))
display(threshold_candidates.style.format(formatters))

## 4. 발표용 그래프

In [ ]:
for filename in [
    'development_class_distribution.png',
    'validation_confusion_matrices.png',
    'validation_roc_pr_curves.png',
]:
    display(Image(filename=str(RESULT_ROOT / 'figures' / filename)))

## 5. 클래스 가중치와 PageValues 실험

같은 공식 validation에서 클래스 가중치와 `PageValues` 특징의 영향을 비교한다.

In [ ]:
from run_ablation import main as run_ablation
run_ablation()
ablation = pd.read_csv(RESULT_ROOT / 'metrics' / 'validation_ablation_metrics.csv')
display(ablation.style.format(formatters))
display(Image(filename=str(RESULT_ROOT / 'figures' / 'validation_ablation_comparison.png')))

## 6. 해석 메모

- 정확도는 미구매 클래스가 많아 단독으로 사용하지 않는다.
- 기준선과 MLP 비교는 동일한 공식 validation 및 전처리에서 수행한다.
- 임계값 후보는 validation 결과이며 비즈니스 비용 합의 전에는 확정하지 않는다.
- `PageValues` 제외 결과로 예측 시점에서 이 특징을 사용할 수 있는지 점검한다.
- 팀의 모델과 임계값을 확정하기 전에는 `--evaluate-test`를 실행하지 않는다.